# 01 — Análise Exploratória dos Espectrogramas (EDA)

Inspeção da distribuição de classes, dimensões das imagens, estatísticas de intensidade por canal RGB e checagem básica de integridade dos espectrogramas em `data/processed/spectrograms/`.

In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

%matplotlib inline
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SPECTROGRAMS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'spectrograms'
CLASSES = ['pre_ictal', 'ictal', 'pos_ictal', 'normal']
COLORS  = {'pre_ictal': '#3498db', 'ictal': '#e74c3c', 'pos_ictal': '#2ecc71', 'normal': '#95a5a6'}

def parse_class(stem: str) -> str:
    for cls in ['pre_ictal', 'pos_ictal', 'normal', 'ictal']:
        if stem.endswith('_' + cls):
            return cls
    return 'unknown'

rows = []
for p in sorted(SPECTROGRAMS_DIR.rglob('*.png')):
    with Image.open(p) as im:
        arr = np.array(im.convert('RGB'), dtype=np.float32)
        w, h, mode = im.width, im.height, im.mode
    rows.append({
        'patient_id': p.parent.name,
        'filename': p.name,
        'class': parse_class(p.stem),
        'width': w,
        'height': h,
        'mode': mode,
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'max': float(arr.max()),
        'mean_R': float(arr[:, :, 0].mean()),
        'mean_G': float(arr[:, :, 1].mean()),
        'mean_B': float(arr[:, :, 2].mean()),
        'md5': hashlib.md5(p.read_bytes()).hexdigest(),
    })

df = pd.DataFrame(rows)
print(f'Total de imagens: {len(df)} | Dimensoes unicas: {df[["width", "height", "mode"]].drop_duplicates().to_dict("records")}')
print(f'Imagens em branco (max==0): {(df["max"] == 0).sum()} | Duplicatas MD5: {df.duplicated("md5").sum()}')
display(df.groupby(['patient_id', 'class']).size().unstack(fill_value=0))

In [ ]:
# Contagem por classe e distribuicao de intensidade media por canal RGB
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df['class'].value_counts().reindex(CLASSES, fill_value=0)
axes[0].bar(CLASSES, counts.values, color=[COLORS[c] for c in CLASSES])
axes[0].set_title('Contagem de Imagens por Classe')
axes[0].set_ylabel('Quantidade')

data_by_cls = [df[df['class'] == c]['mean'].values for c in CLASSES]
bp = axes[1].boxplot(data_by_cls, labels=CLASSES, patch_artist=True)
for patch, c in zip(bp['boxes'], CLASSES):
    patch.set_facecolor(COLORS[c])
axes[1].set_title('Intensidade Media Global por Classe')
axes[1].set_ylabel('Intensidade Media [0-255]')

plt.tight_layout()
plt.show()

display(df.groupby('class')[['mean', 'std', 'mean_R', 'mean_G', 'mean_B']].mean().reindex(CLASSES).round(2))